# CVM DATA MINER
## 1. Libraries and constants

In [111]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import zipfile
import io
from datetime import date
from pathlib import Path
DATA_DIR = Path("../data")
DFP_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/"
CAPEX_TERMS = [
    "imobiliz",
    "intang",
    "software",
    "ativos? de contrato",
    "ativos? contratua",
    "infraestrutura",
    "ativos? da concess",
    "servi.o p.blico",
    "biol",
    "propriedades? para investimento",
]

EXCLUDE_TERMS = [
    "venda",
    "aliena",
    "baixa",
    "receb",
    "indeniza",
    "adiantamento",
    "transf",
    "decr.scimo",
]

CAPEX_KEYWORDS = "|".join(CAPEX_TERMS)
EXCLUDE_KEYWORDS = "|".join(EXCLUDE_TERMS)
pd.set_option("display.max_rows", None)

## 2. Functions

In [112]:
def find_zips(url):
    """Return a dictionary mapping each year to its zip URL."""
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    tags = soup.find_all("a")
    zip_dict = {}
    for tag in tags:
        href = tag.get("href")
        if not href:
            continue
        if href.endswith(".zip"):
            year = int(href.removesuffix(".zip")[-4:])
            link = urljoin(url, href)
            zip_dict[year] = link
    return zip_dict

def complete_years(zip_links):
    """Return only the years whose DFP files are complete.

    Excludes the current year and, before April, also the previous
    year, since companies have until the end of March to file.
    """
    today = date.today()
    complete_year_links = {}
    for year, link in zip_links.items():
        if (year == today.year - 1 and today.month < 4) or (year >= today.year):
            continue
        complete_year_links[year] = link
    return complete_year_links

def download_zip(zip_link):
    """Return a zip as an in-memory buffer, downloading it only if it is not cached in DATA_DIR."""
    DATA_DIR.mkdir(exist_ok=True)
    file_name = zip_link.split("/")[-1]
    file_path = DATA_DIR / file_name
    if not file_path.exists():
        response = requests.get(zip_link, timeout=60)
        response.raise_for_status()
        file_path.write_bytes(response.content)    
    return io.BytesIO(file_path.read_bytes())

def open_csv(zip_file, statement, basis, year, doc_type="dfp"):
    """Read a statement CSV from a CVM zip (DFP or ITR) and return a DataFrame."""
    with zipfile.ZipFile(zip_file) as z:
        csv_name = f"{doc_type}_cia_aberta_{statement}_{basis}_{year}.csv"
        df = pd.read_csv(z.open(csv_name), sep=";", encoding="latin-1")
    return df

def split_by_company(df):
    """Clean a CVM statement and split it by company.

    Keeps the latest fiscal year, drops duplicated rows and converts
    values to thousands of reais. Returns a dictionary mapping each
    CD_CVM to its DataFrame.
    """
    df_clean = df[(df["ORDEM_EXERC"] == "ÚLTIMO") & (~df.duplicated())]
    is_unit = df_clean["ESCALA_MOEDA"] == "UNIDADE"
    df_clean.loc[is_unit, "VL_CONTA"] /= 1000
    df_clean.loc[is_unit, "ESCALA_MOEDA"] = "MIL"
    companies_data = {}
    for code, group in df_clean.groupby("CD_CVM"):
        companies_data[code] = group
    return companies_data

def find_capex(company):
    """Return the total capex of one company.

    Values are in thousands of reais (negative = cash out).
    """
    investment = company[company["CD_CONTA"].str.startswith("6.02.")]
    is_asset = investment["DS_CONTA"].str.contains(CAPEX_KEYWORDS, case=False)
    is_sale = investment["DS_CONTA"].str.contains(EXCLUDE_KEYWORDS, case=False)
    capex = investment[is_asset & ~is_sale]
    return capex["VL_CONTA"].sum()

def find_cfo(company):
    """Return the operating cash flow (account 6.01) of one company.

    Values are in thousands of reais. Returns NaN if the account is
    missing and raises ValueError if it appears more than once.
    """
    is_cfo = company["CD_CONTA"] == "6.01"
    cfo = company[is_cfo]
    cfo_value = cfo["VL_CONTA"]
    if len(cfo_value) == 0:
        return float("nan")
    elif len(cfo_value) == 1:
        return cfo_value.iloc[0]
    else:
        code = company["CD_CVM"].iloc[0]
        raise ValueError(f"Company {code} has {len(cfo_value)} rows for account 6.01")

def build_fcf_table(companies_data, year):
    """Build a table with the free cash flow of each company in one year.

    Returns a DataFrame with one row per company and the columns
    CD_CVM, DENOM_CIA, YEAR, CFO, CAPEX and FCF. Values are in
    thousands of reais; FCF = CFO + CAPEX (capex is negative).
    """
    rows = []
    for code, group in companies_data.items():
        cfo = find_cfo(group)
        capex = find_capex(group)
        rows.append({
            "CD_CVM": code, "DENOM_CIA": group["DENOM_CIA"].iloc[0], "YEAR": year,
            "CFO": cfo, "CAPEX": capex, "FCF": cfo + capex})
    return pd.DataFrame(rows)

def build_fcf_history(dfp_links):
    """Build the free cash flow history of all companies.

    Takes a dictionary mapping each year to its DFP zip URL and returns
    a single DataFrame with one row per company and year, with the
    columns CD_CVM, DENOM_CIA, YEAR, CFO, CAPEX and FCF (in thousands
    of reais).
    """
    tables = []
    for year, link in dfp_links.items():
        zip_data = download_zip(link)
        df = open_csv(zip_data, "DFC_MI", "con", year, "dfp")
        companies_data = split_by_company(df)
        fcf_table = build_fcf_table(companies_data, year)
        tables.append(fcf_table)
    return pd.concat(tables, ignore_index=True)

In [113]:
year = 2025
statement = "DFC_MI"
basis = "con"
doc_type = "dfp"
zip_links = find_zips(DFP_URL)
dfp_links = complete_years(zip_links)
zip_data = download_zip(zip_links[year])
df = open_csv(zip_data, statement, basis, year, doc_type)

In [115]:
companies_data = split_by_company(df)
test = companies_data[24805]
print(find_capex(test))

-80443.288


In [ ]:
cmig = companies_data[2453]
